# Hafta 10 — Geri Yayılım ve PyTorch'a Giriş

Önce türevleri elle ve NumPy ile alıyoruz, sonra PyTorch'un autograd'ı ile doğruluyoruz; sonunda ilk gerçek eğitim döngüsü. Veri: `elektrik_tuketimi.csv`.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, torch.nn as nn
torch.manual_seed(0); np.random.seed(0)
sig = lambda z: 1/(1+np.exp(-z)); relu = lambda z: np.maximum(z, 0)
print("PyTorch", torch.__version__, "| GPU var mı?", torch.cuda.is_available())

## 1. Tek nöron: gradyan elle (Örnek 10.1)

In [ ]:
x, w, b, y, eta = 2.0, 0.5, -0.5, 1, 0.1
for adim in range(5):
    z = w*x + b; a = sig(z); L = -(y*np.log(a) + (1-y)*np.log(1-a))
    dz = a - y; dw, db = dz*x, dz
    print(f"adım {adim}: w={w:.3f} b={b:.3f} a={a:.3f} L={L:.3f}  dL/dw={dw:.3f} dL/db={db:.3f}")
    w, b = w - eta*dw, b - eta*db

## 2. Aynı hesabı autograd ile

In [ ]:
w = torch.tensor(0.5, requires_grad=True); b = torch.tensor(-0.5, requires_grad=True)
xt, yt = torch.tensor(2.0), torch.tensor(1.0)
a = torch.sigmoid(w*xt + b); L = -(yt*torch.log(a) + (1-yt)*torch.log(1-a))
L.backward()
print("autograd:", w.grad.item(), b.grad.item(), " (elle: -0.756, -0.378)")

## 3. 2-2-1 ağda geri yayılım: NumPy (Örnek 10.2) ve autograd doğrulaması

In [ ]:
x = np.array([1., 2.]); W1 = np.array([[.5, -1], [1, .5]]); b1 = np.array([0., -1]); W2 = np.array([1., -1]); b2 = 0.; y = 0
# ileri
z1 = W1 @ x + b1; a1 = relu(z1); z2 = W2 @ a1 + b2; yp = sig(z2); L = -(y*np.log(yp) + (1-y)*np.log(1-yp))
# geri
d2 = yp - y; dW2 = d2 * a1; db2 = d2
d1 = (W2 * d2) * (z1 > 0); dW1 = np.outer(d1, x); db1 = d1
print(f"ŷ={yp:.3f} L={L:.3f}"); print("dW2", dW2.round(3), " db2", round(db2, 3)); print("dW1", dW1.round(3)); print("db1", db1.round(3))

In [ ]:
tW1 = torch.tensor(W1, requires_grad=True); tb1 = torch.tensor(b1, requires_grad=True); tW2 = torch.tensor(W2, requires_grad=True); tb2 = torch.tensor(b2, requires_grad=True)
tx = torch.tensor(x); ta1 = torch.relu(tW1 @ tx + tb1); typ = torch.sigmoid(tW2 @ ta1 + tb2); tL = -torch.log(1 - typ)
tL.backward()
print("autograd dW1:\n", tW1.grad.numpy().round(3)); print("autograd db1:", tb1.grad.numpy().round(3), " dW2:", tW2.grad.numpy().round(3), " db2:", round(tb2.grad.item(), 3))
print("eşleşiyor mu?", np.allclose(tW1.grad.numpy(), dW1), np.allclose(tW2.grad.numpy(), dW2))

## 4. Genel geri yayılım: 7-16-1 regresyon ağı NumPy ile (tam batch)

In [ ]:
# Dersin yük verisi -> 7 özellik (5. haftadaki gibi), zamana göre ayrım, ölçekleme
df = pd.read_csv("elektrik_tuketimi.csv"); df = df[df.tuketim_kW < 300].copy(); df["sicaklik_C"] = df["sicaklik_C"].interpolate()
h = 2*np.pi*df.saat.values/24
X = np.c_[np.sin(h), np.cos(h), np.sin(2*h), np.cos(2*h), df.hafta_sonu, df.sicaklik_C, np.clip(df.sicaklik_C - 24, 0, None)]; y = df.tuketim_kW.values
tr = df.gun.values <= 24; te = ~tr
mu, sd = X[tr].mean(0), X[tr].std(0); Xs = (X - mu)/sd; ym, ysd = y[tr].mean(), y[tr].std(); ys = (y - ym)/ysd
X_tr, y_tr = torch.tensor(Xs[tr], dtype=torch.float32), torch.tensor(ys[tr], dtype=torch.float32)[:, None]
X_te, y_te = torch.tensor(Xs[te], dtype=torch.float32), torch.tensor(ys[te], dtype=torch.float32)[:, None]
print(X_tr.shape, X_te.shape)

In [ ]:
def numpy_ag_egit(Xn, yn, gizli=16, eta=0.05, epoch=300, seed=0):
    rng = np.random.default_rng(seed); d = Xn.shape[1]
    W1 = rng.normal(0, np.sqrt(2/d), (gizli, d)); b1 = np.zeros(gizli); W2 = rng.normal(0, np.sqrt(1/gizli), (1, gizli)); b2 = np.zeros(1); hist = []
    for ep in range(epoch):
        Z1 = Xn @ W1.T + b1; A1 = relu(Z1); Yp = A1 @ W2.T + b2                  # ileri (N×gizli, N×1); çıktı doğrusal (regresyon)
        E = Yp - yn; L = np.mean(E**2); hist.append(L)
        d2 = 2*E/len(yn)                                                          # ∂L/∂z2 (MSE)
        dW2 = d2.T @ A1; db2 = d2.sum(0)
        d1 = (d2 @ W2) * (Z1 > 0)                                                 # geriye taşı ⊙ ReLU'
        dW1 = d1.T @ Xn; db1 = d1.sum(0)
        W1 -= eta*dW1; b1 -= eta*db1; W2 -= eta*dW2; b2 -= eta*db2
    return (W1, b1, W2, b2), hist
param, hist = numpy_ag_egit(Xs[tr], ys[tr][:, None])
W1, b1, W2, b2 = param; tahmin = (relu(Xs[te] @ W1.T + b1) @ W2.T + b2).ravel()*ysd + ym
print("NumPy ağ test RMSE (kW):", np.sqrt(np.mean((tahmin - y[te])**2)).round(2), " (5. hafta doğrusal: ~11.8)")
plt.plot(hist); plt.xlabel("epoch"); plt.ylabel("MSE (ölçekli)"); plt.yscale("log"); plt.grid(alpha=.3); plt.show()

## 5. PyTorch eğitim döngüsü

In [ ]:
torch.manual_seed(0)
model = nn.Sequential(nn.Linear(7, 32), nn.ReLU(), nn.Linear(32, 1))
kayip = nn.MSELoss(); opt = torch.optim.Adam(model.parameters(), lr=1e-2)
h_tr, h_te = [], []
for epoch in range(300):
    model.train(); opt.zero_grad(); L = kayip(model(X_tr), y_tr); L.backward(); opt.step(); h_tr.append(L.item())
    model.eval()
    with torch.no_grad(): h_te.append(kayip(model(X_te), y_te).item())
plt.plot(h_tr, label="eğitim"); plt.plot(h_te, label="test"); plt.yscale("log"); plt.legend(); plt.xlabel("epoch"); plt.grid(alpha=.3); plt.show()
with torch.no_grad(): tah = model(X_te).numpy().ravel()*ysd + ym
print("PyTorch ağ test RMSE (kW):", np.sqrt(np.mean((tah - y[te])**2)).round(2))

## 6. Mini-batch ile DataLoader

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
torch.manual_seed(0); model = nn.Sequential(nn.Linear(7, 32), nn.ReLU(), nn.Linear(32, 1)); opt = torch.optim.Adam(model.parameters(), lr=1e-2)
yukleyici = DataLoader(TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True)
for epoch in range(40):
    for xb, yb in yukleyici:
        opt.zero_grad(); L = kayip(model(xb), yb); L.backward(); opt.step()
    if epoch % 10 == 0:
        with torch.no_grad(): print(f"epoch {epoch:2d}  eğitim {kayip(model(X_tr), y_tr).item():.4f}  test {kayip(model(X_te), y_te).item():.4f}   ({len(yukleyici)} adım/epoch)")

## 7. Öğrenme oranı ve optimizer karşılaştırması

In [ ]:
def egit(opt_fn, lr, epoch=150):
    torch.manual_seed(0); m = nn.Sequential(nn.Linear(7, 32), nn.ReLU(), nn.Linear(32, 1)); o = opt_fn(m.parameters(), lr=lr); hist = []
    for _ in range(epoch):
        o.zero_grad(); L = kayip(m(X_tr), y_tr); L.backward(); o.step(); hist.append(min(L.item(), 10))
    return hist
fig, ax = plt.subplots(1, 3, figsize=(14, 3.3))
for a, (ad, fn) in zip(ax, (("SGD", torch.optim.SGD), ("SGD+momentum", lambda p, lr: torch.optim.SGD(p, lr=lr, momentum=0.9)), ("Adam", torch.optim.Adam))):
    for lr in (1e-3, 1e-2, 1e-1): a.plot(egit(fn, lr), label=f"η={lr}")
    a.set_yscale("log"); a.set_title(ad); a.legend(fontsize=8); a.grid(alpha=.3); a.set_xlabel("epoch")
plt.tight_layout(); plt.show()

## 8. Kaybolan gradyanı ölçmek

In [ ]:
for akt in (nn.Sigmoid, nn.ReLU):
    torch.manual_seed(0); katman = []
    for i in range(8): katman += [nn.Linear(7 if i == 0 else 32, 32), akt()]
    m = nn.Sequential(*katman, nn.Linear(32, 1)); L = kayip(m(X_tr), y_tr); L.backward()
    normlar = [p.grad.norm().item() for n, p in m.named_parameters() if "weight" in n]
    print(akt.__name__, "katman başına ‖grad‖:", np.array(normlar).round(6))

## 9. Alıştırmalar

**Alıştırma 1.** Örnek 10.2'yi y = 1 için tekrarlayın (elle + kod). Hangi gradyanların işareti değişti, neden?

In [ ]:
# Alıştırma 1

**Alıştırma 2.** `numpy_ag_egit` fonksiyonuna ikinci bir gizli katman ekleyin (7-16-8-1). Geri yayılımda hangi satırlar eklendi? Test RMSE değişti mi?

In [ ]:
# Alıştırma 2

**Alıştırma 3.** PyTorch modelinde `opt.zero_grad()` satırını silip 20 epoch eğitin; kayıp ne yapıyor? İlk 3 adımda `model[0].weight.grad.norm()` değerini yazdırıp neden büyüdüğünü açıklayın.

In [ ]:
# Alıştırma 3

**Alıştırma 4.** Batch büyüklüğü 8, 32, 128, 575 (tam) için 30 epoch eğitin; her biri için toplam adım sayısı, süre (`time.time()`) ve son test kaybını tablo yapın.

In [ ]:
# Alıştırma 4

**Alıştırma 5.** Sigmoid + MSE ile sigmoid + log-loss'u ikili sınıflandırmada karşılaştırın: motor verisinde (5 özellik) tek nöronlu PyTorch modeli, her iki kayıpla 200 epoch, η = 0.1 SGD; kayıp ve test AUC eğrileri. Doygunluk (a ≈ 0 veya 1) MSE'de öğrenmeyi nasıl yavaşlatıyor? (ipucu: ∂L/∂z = 2(a−y)a(1−a) vs (a−y))

In [ ]:
# Alıştırma 5